# Chapter 6 — The Fourier Transform and Its Applications

This notebook turns the chapter into an interactive laboratory. It uses the convention

$$
\widehat f(\omega)=\int_{-\infty}^{\infty}f(t)e^{-i\omega t}\,dt,
\qquad
f(t)=\frac1{2\pi}\int_{-\infty}^{\infty}\widehat f(\omega)e^{i\omega t}\,d\omega.
$$

The experiments cover:

- $L^1$ and $L^2$ behavior on the real line;
- fundamental transform pairs;
- translation, modulation, scaling, differentiation, and convolution;
- inversion, Plancherel's identity, and uncertainty;
- Gaussian approximate identities and heat flow;
- characteristic functions, independent sums, moments, and the central limit theorem;
- differential and integral equations;
- formal transform calculations followed by **direct verification** of candidate solutions.

> Run the notebook from top to bottom. Before pressing a button, predict the effect in both the physical and frequency domains.


## 1. Computational convention and numerical transform

On a finite grid $t_j=-T+j\Delta t$, the transform integral is approximated by

$$
\widehat f(\omega_k)
\approx \Delta t\sum_j f(t_j)e^{-i\omega_k t_j},
\qquad
\omega_k=\frac{2\pi k}{N\Delta t}.
$$

The discrete FFT makes this calculation efficient, but it introduces two approximations:

1. truncation of the infinite line to a finite interval;
2. sampling of both the function and its spectrum.

The helper functions below include the factors required by the chapter's normalization.


In [ ]:
# Core numerical and interactive tools.
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 4.8), "font.size": 11})

PI = np.pi
TWO_PI = 2.0 * PI


def time_grid(half_width=20.0, samples=2**14):
    """Return an equally spaced symmetric grid with its right endpoint omitted."""
    return np.linspace(-half_width, half_width, samples, endpoint=False)


def continuous_fft(t, values):
    """Approximate F(omega)=integral f(t) exp(-i omega t) dt."""
    t = np.asarray(t)
    values = np.asarray(values)
    dt = t[1] - t[0]
    omega = TWO_PI * np.fft.fftshift(np.fft.fftfreq(t.size, d=dt))
    transform = dt * np.fft.fftshift(np.fft.fft(np.fft.ifftshift(values)))
    return omega, transform


def continuous_ifft(omega, transform):
    """Invert a spectrum sampled on the matching FFT frequency grid."""
    omega = np.asarray(omega)
    transform = np.asarray(transform)
    domega = omega[1] - omega[0]
    values = (transform.size * domega / TWO_PI) * np.fft.fftshift(
        np.fft.ifft(np.fft.ifftshift(transform))
    )
    return values


def integrate_grid(x, values):
    """Trapezoidal integration on an ordered numerical grid."""
    return np.trapezoid(values, x)


def l2_norm_grid(x, values):
    """Approximate an L2 norm on the displayed finite interval."""
    return np.sqrt(np.real(integrate_grid(x, np.abs(values) ** 2)))


display(HTML(
    "<div style='padding:10px;border-left:4px solid #2a6fbb;background:#eef6ff'>"
    "<b>Setup complete.</b> The continuous-transform FFT convention is ready."
    "</div>"
))


## 2. $L^1$ and $L^2$ are different on the real line

Consider

$$
f_p(t)=\frac1{(1+|t|)^p}.
$$

Its tail behaves like $|t|^{-p}$, so

$$
f_p\in L^1(\mathbb R)\iff p>1,
\qquad
f_p\in L^2(\mathbb R)\iff 2p>1.
$$

Thus $L^1(\mathbb R)$ and $L^2(\mathbb R)$ do not contain one another. The experiment plots the truncated quantities

$$
\int_{-R}^{R}|f_p(t)|\,dt,
\qquad
\left(\int_{-R}^{R}|f_p(t)|^2\,dt\right)^{1/2}
$$

as $R$ grows.


In [ ]:
# Diagnose integrability by observing truncated norm growth.
lp_power = widgets.FloatSlider(value=0.75, min=0.25, max=1.75, step=0.05, description="p:")
lp_radius = widgets.IntSlider(value=100, min=10, max=500, step=10, description="max R:")
lp_button = widgets.Button(description="Test L1 and L2", button_style="primary", icon="search")
lp_output = widgets.Output()


def explore_lp_membership(_=None):
    """Plot truncated L1 and L2 quantities for the algebraic-tail family."""
    with lp_output:
        clear_output(wait=True)
        p = lp_power.value
        radii = np.geomspace(1.0, lp_radius.value, 90)

        # These closed forms avoid quadrature error for very large intervals.
        if abs(p - 1.0) < 1e-12:
            l1_values = 2.0 * np.log1p(radii)
        else:
            l1_values = 2.0 * ((1.0 + radii) ** (1.0 - p) - 1.0) / (1.0 - p)

        if abs(2 * p - 1.0) < 1e-12:
            l2_squared = 2.0 * np.log1p(radii)
        else:
            l2_squared = 2.0 * ((1.0 + radii) ** (1.0 - 2 * p) - 1.0) / (1.0 - 2 * p)
        l2_values = np.sqrt(l2_squared)

        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
        axes[0].semilogx(radii, l1_values, color="#1565c0")
        axes[0].set_title("Truncated L1 norm")
        axes[0].set_xlabel("R")
        axes[1].semilogx(radii, l2_values, color="#d95f02")
        axes[1].set_title("Truncated L2 norm")
        axes[1].set_xlabel("R")
        plt.show()

        l1_status = "belongs to L1" if p > 1 else "does not belong to L1"
        l2_status = "belongs to L2" if p > 0.5 else "does not belong to L2"
        display(HTML(f"<b>Theoretical diagnosis:</b> f_p {l1_status} and {l2_status}."))


lp_button.on_click(explore_lp_membership)
display(widgets.VBox([widgets.HBox([lp_power, lp_radius]), lp_button, lp_output]))
explore_lp_membership()


## 3. Fundamental transform pairs

The chapter's basic examples include

$$
\begin{array}{c|c}
f(t)&\widehat f(\omega)\\ \hline
\mathbf 1_{[-a,a]}(t)&\displaystyle \frac{2\sin(a\omega)}{\omega}\\[2mm]
(1-|t|/a)_+&\displaystyle a\left(\frac{\sin(a\omega/2)}{a\omega/2}\right)^2\\[2mm]
e^{-a|t|}&\displaystyle \frac{2a}{a^2+\omega^2}\\[2mm]
e^{-at}\mathbf 1_{[0,\infty)}(t)&\displaystyle \frac1{a+i\omega}\\[2mm]
e^{-at^2}&\displaystyle \sqrt{\frac\pi a}\,e^{-\omega^2/(4a)}.
\end{array}
$$

The Riemann–Lebesgue lemma predicts $\widehat f(\omega)\to0$ for every $f\in L^1$. Compare the numerical FFT with the exact formula.


In [ ]:
# Interactive gallery of exact and numerical Fourier-transform pairs.
pair_name = widgets.Dropdown(
    options=["Rectangular pulse", "Triangular pulse", "Two-sided exponential", "One-sided exponential", "Gaussian"],
    value="Rectangular pulse",
    description="Pair:",
)
pair_parameter = widgets.FloatSlider(value=1.0, min=0.25, max=3.0, step=0.05, description="a:")
pair_button = widgets.Button(description="Compute transform", button_style="info", icon="wave-square")
pair_output = widgets.Output()


def transform_pair(name, t, omega, a):
    """Return a time-domain function and its exact transform."""
    if name == "Rectangular pulse":
        f = (np.abs(t) <= a).astype(float)
        exact = 2 * a * np.sinc(a * omega / PI)
    elif name == "Triangular pulse":
        f = np.maximum(1.0 - np.abs(t) / a, 0.0)
        exact = a * np.sinc(a * omega / (2 * PI)) ** 2
    elif name == "Two-sided exponential":
        f = np.exp(-a * np.abs(t))
        exact = 2 * a / (a * a + omega * omega)
    elif name == "One-sided exponential":
        f = np.where(t >= 0, np.exp(-a * t), 0.0)
        exact = 1.0 / (a + 1j * omega)
    else:
        f = np.exp(-a * t * t)
        exact = np.sqrt(PI / a) * np.exp(-omega * omega / (4 * a))
    return f, exact


def show_transform_pair(_=None):
    """Plot the time function and compare exact and numerical spectra."""
    with pair_output:
        clear_output(wait=True)
        t = time_grid(half_width=24.0, samples=2**15)
        omega_template = TWO_PI * np.fft.fftshift(np.fft.fftfreq(t.size, d=t[1] - t[0]))
        f, exact = transform_pair(pair_name.value, t, omega_template, pair_parameter.value)
        omega, numerical = continuous_fft(t, f)
        central = np.abs(omega) <= 18
        error = np.max(np.abs(numerical[central] - exact[central]))

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, np.real(f), color="#1565c0")
        axes[0].set_xlim(-6, 6)
        axes[0].set_title("Time domain")
        axes[1].plot(omega[central], np.abs(numerical[central]), color="#d95f02", label="numerical magnitude")
        axes[1].plot(omega[central], np.abs(exact[central]), color="black", linestyle="--", label="exact magnitude")
        axes[1].set_title("Frequency domain")
        axes[1].legend()
        plt.show()

        display(HTML(f"<b>Maximum central-spectrum error:</b> {error:.3e}"))


pair_button.on_click(show_transform_pair)
display(widgets.VBox([widgets.HBox([pair_name, pair_parameter]), pair_button, pair_output]))
show_transform_pair()


## 4. Enter a function and compute its Fourier transform

The user may enter a function of $t$ and ask for the symbolic transform

$$
\widehat f(\omega)
=\int_{-\infty}^{\infty}f(t)e^{-i\omega t}\,dt.
$$

The input uses SymPy syntax. Typical examples are:

- <code>exp(-t**2)</code> for $e^{-t^2}$;
- <code>exp(-Abs(t))</code> for $e^{-|t|}$;
- <code>exp(-2*t)*Heaviside(t)</code> for a one-sided exponential;
- <code>Piecewise((1, Abs(t) <= 1), (0, True))</code> for a rectangular pulse;
- <code>exp(-a*t**2)</code> with the built-in assumption $a>0$.

The cell displays the entered function, the defining integral, and the simplified answer using MathJax. If no closed form is found, it keeps the unevaluated integral visible instead of presenting a misleading formula.


In [ ]:
# Symbolic Fourier-transform calculator with MathJax output.
try:
    import sympy as sp
except ImportError as exc:
    raise ImportError(
        "This cell requires SymPy. Install it with %pip install sympy and run the cell again."
    ) from exc

from IPython.display import Math

# Real transform variables and positive parameters available to the user.
t_symbol = sp.symbols("t", real=True)
omega_symbol = sp.symbols("omega", real=True)
a_symbol, b_symbol = sp.symbols("a b", positive=True, real=True)

allowed_names = {
    "t": t_symbol,
    "a": a_symbol,
    "b": b_symbol,
    "exp": sp.exp,
    "sin": sp.sin,
    "cos": sp.cos,
    "sinh": sp.sinh,
    "cosh": sp.cosh,
    "Abs": sp.Abs,
    "sqrt": sp.sqrt,
    "log": sp.log,
    "Heaviside": sp.Heaviside,
    "Piecewise": sp.Piecewise,
    "pi": sp.pi,
    "I": sp.I,
    "oo": sp.oo,
    "True": True,
    "False": False,
}

transform_examples = {
    "Gaussian": "exp(-t**2)",
    "Parameterized Gaussian": "exp(-a*t**2)",
    "Two-sided exponential": "exp(-Abs(t))",
    "One-sided exponential": "exp(-2*t)*Heaviside(t)",
    "Rectangular pulse": "Piecewise((1, Abs(t) <= 1), (0, True))",
    "Custom expression": "",
}

custom_example = widgets.Dropdown(
    options=list(transform_examples),
    value="Gaussian",
    description="Example:",
    layout=widgets.Layout(width="55%"),
)
custom_function = widgets.Text(
    value=transform_examples["Gaussian"],
    description="f(t):",
    placeholder="Enter a SymPy expression in t",
    layout=widgets.Layout(width="85%"),
)
custom_compute = widgets.Button(
    description="Compute Fourier transform",
    button_style="primary",
    icon="calculator",
)
custom_output = widgets.Output()


def load_transform_example(change):
    """Copy a selected preset into the editable input field."""
    expression = transform_examples[change["new"]]
    if expression:
        custom_function.value = expression


def compute_custom_transform(_=None):
    """Parse the user's function, integrate it, and display MathJax formulas."""
    with custom_output:
        clear_output(wait=True)
        source = custom_function.value.strip()
        if not source:
            display(HTML(
                "<div style='padding:10px;color:#842029;background:#f8d7da'>"
                "Please enter a function of <b>t</b>.</div>"
            ))
            return

        try:
            expression = sp.sympify(source, locals=allowed_names)
            extra_symbols = expression.free_symbols - {t_symbol, a_symbol, b_symbol}
            if extra_symbols:
                names = ", ".join(sorted(str(symbol) for symbol in extra_symbols))
                raise ValueError(f"Unsupported symbols: {names}. Use only t and the positive parameters a, b.")

            integrand = expression * sp.exp(-sp.I * omega_symbol * t_symbol)
            result = sp.integrate(
                integrand,
                (t_symbol, -sp.oo, sp.oo),
                conds="piecewise",
            )
            result = sp.simplify(result)

            display(HTML(
                "<div style='padding:10px;border-left:4px solid #2a6fbb;background:#eef6ff'>"
                "<b>Input interpreted successfully.</b> "
                "The symbols <i>a</i> and <i>b</i>, when used, are assumed positive."
                "</div>"
            ))
            display(Math(r"f(t)=" + sp.latex(expression)))
            display(Math(
                r"\widehat f(\omega)="
                r"\int_{-\infty}^{\infty}\left("
                + sp.latex(expression)
                + r"\right)e^{-i\omega t}\,dt"
            ))

            if result.has(sp.Integral):
                display(HTML(
                    "<div style='padding:10px;color:#664d03;background:#fff3cd'>"
                    "<b>No closed form was found under the current assumptions.</b> "
                    "The exact unevaluated integral is shown below."
                    "</div>"
                ))
                display(Math(r"\widehat f(\omega)=" + sp.latex(result)))
            else:
                display(HTML(
                    "<div style='padding:10px;color:#0f5132;background:#d1e7dd'>"
                    "<b>Closed-form transform found.</b>"
                    "</div>"
                ))
                display(Math(
                    r"\boxed{\widehat f(\omega)="
                    + sp.latex(result)
                    + r"}"
                ))

        except Exception as error:
            display(HTML(
                "<div style='padding:10px;color:#842029;background:#f8d7da'>"
                "<b>The expression could not be transformed.</b><br>"
                + str(error)
                + "<br><br>Try syntax such as <code>exp(-t**2)</code>, "
                "<code>exp(-Abs(t))</code>, or choose a preset."
                "</div>"
            ))


custom_example.observe(load_transform_example, names="value")
custom_compute.on_click(compute_custom_transform)
display(widgets.VBox([
    custom_example,
    custom_function,
    custom_compute,
    custom_output,
]))
compute_custom_transform()


## 5. Translation, modulation, and scaling

For $f\in L^1(\mathbb R)$,

$$
\begin{aligned}
f(t-t_0)&\longleftrightarrow e^{-i\omega t_0}\widehat f(\omega),\\
e^{i\omega_0t}f(t)&\longleftrightarrow \widehat f(\omega-\omega_0),\\
f(at)&\longleftrightarrow \frac1{|a|}\widehat f\left(\frac\omega a\right).
\end{aligned}
$$

Translation changes phase, modulation shifts the spectrum, and compression in time expands the spectrum. The Gaussian baseline makes each prediction visually clear.


In [ ]:
# Visualize three elementary transform rules using a Gaussian baseline.
rule_choice = widgets.ToggleButtons(options=["Translation", "Modulation", "Scaling"], description="Rule:")
rule_value = widgets.FloatSlider(value=1.5, min=0.25, max=4.0, step=0.05, description="Parameter:")
rule_button = widgets.Button(description="Apply rule", button_style="primary", icon="exchange-alt")
rule_output = widgets.Output()


def apply_transform_rule(_=None):
    """Compare the numerical transformed signal with the exact rule prediction."""
    with rule_output:
        clear_output(wait=True)
        t = time_grid(half_width=18.0, samples=2**14)
        dt = t[1] - t[0]
        omega = TWO_PI * np.fft.fftshift(np.fft.fftfreq(t.size, d=dt))
        baseline = np.exp(-t * t)
        baseline_exact = np.sqrt(PI) * np.exp(-omega * omega / 4)
        value = rule_value.value

        if rule_choice.value == "Translation":
            changed = np.exp(-(t - value) ** 2)
            predicted = np.exp(-1j * omega * value) * baseline_exact
            explanation = "Magnitude unchanged; phase acquires -omega*t0."
        elif rule_choice.value == "Modulation":
            changed = np.exp(1j * value * t) * baseline
            predicted = np.sqrt(PI) * np.exp(-(omega - value) ** 2 / 4)
            explanation = "The spectrum is shifted by omega0."
        else:
            changed = np.exp(-(value * t) ** 2)
            predicted = np.sqrt(PI) / value * np.exp(-(omega / value) ** 2 / 4)
            explanation = "Time compression produces frequency expansion."

        _, numerical = continuous_fft(t, changed)
        central = np.abs(omega) <= 12
        error = np.max(np.abs(numerical[central] - predicted[central]))

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, np.real(baseline), color="black", linestyle="--", label="baseline")
        axes[0].plot(t, np.real(changed), color="#1565c0", label="changed signal")
        axes[0].set_xlim(-7, 7)
        axes[0].legend()
        axes[1].plot(omega[central], np.abs(numerical[central]), color="#d95f02", label="numerical")
        axes[1].plot(omega[central], np.abs(predicted[central]), color="black", linestyle="--", label="predicted")
        axes[1].legend()
        plt.show()
        display(HTML(f"<b>{explanation}</b><br>Maximum identity error: {error:.3e}"))


rule_button.on_click(apply_transform_rule)
display(widgets.VBox([rule_choice, rule_value, rule_button, rule_output]))
apply_transform_rule()


## 6. Differentiation and multiplication by $t$

Under the stated integrability and regularity hypotheses,

$$
\widehat{f'}(\omega)=i\omega\widehat f(\omega),
\qquad
\widehat{tf(t)}(\omega)=i\frac{d}{d\omega}\widehat f(\omega).
$$

These rules exchange differentiation with multiplication. They are the main algebraic mechanism behind transform methods for differential equations.

For $f(t)=e^{-t^2}$,

$$
f'(t)=-2te^{-t^2},
\qquad
\widehat f(\omega)=\sqrt\pi e^{-\omega^2/4}.
$$


In [ ]:
# Verify differentiation and multiplication rules for a Gaussian.
differential_rule = widgets.ToggleButtons(options=["Derivative", "Multiply by t"], description="Identity:")
differential_button = widgets.Button(description="Verify rule", button_style="success", icon="check")
differential_output = widgets.Output()


def verify_differential_rule(_=None):
    """Compare a transformed Gaussian expression with its exact prediction."""
    with differential_output:
        clear_output(wait=True)
        t = time_grid(half_width=16.0, samples=2**14)
        dt = t[1] - t[0]
        omega = TWO_PI * np.fft.fftshift(np.fft.fftfreq(t.size, d=dt))
        f_hat = np.sqrt(PI) * np.exp(-omega * omega / 4)

        if differential_rule.value == "Derivative":
            signal = -2 * t * np.exp(-t * t)
            predicted = 1j * omega * f_hat
            formula = "F{f'} = i omega F{f}"
        else:
            signal = t * np.exp(-t * t)
            predicted = -0.5j * omega * f_hat
            formula = "F{t f} = i dF/domega"

        _, numerical = continuous_fft(t, signal)
        central = np.abs(omega) <= 10
        error = np.max(np.abs(numerical[central] - predicted[central]))

        fig, ax = plt.subplots()
        ax.plot(omega[central], np.imag(numerical[central]), color="#1565c0", label="numerical imaginary part")
        ax.plot(omega[central], np.imag(predicted[central]), color="black", linestyle="--", label="exact prediction")
        ax.set_title(formula)
        ax.legend()
        plt.show()
        display(HTML(f"<b>Maximum central-spectrum error:</b> {error:.3e}"))


differential_button.on_click(verify_differential_rule)
display(widgets.VBox([differential_rule, differential_button, differential_output]))
verify_differential_rule()


## 7. Convolution theorem

For

$$
(f*g)(t)=\int_{-\infty}^{\infty}f(t-s)g(s)\,ds,
$$

the transform converts convolution into multiplication:

$$
\widehat{f*g}(\omega)=\widehat f(\omega)\widehat g(\omega).
$$

For two Gaussians,

$$
e^{-at^2}*e^{-bt^2}
=\sqrt{\frac\pi{a+b}}
\exp\left(-\frac{ab}{a+b}t^2\right).
$$

The exact formula provides a strong numerical check of both convolution and normalization.


In [ ]:
# Convolve two Gaussians and verify the transform product rule.
conv_a = widgets.FloatSlider(value=1.0, min=0.25, max=3.0, step=0.05, description="a:")
conv_b = widgets.FloatSlider(value=0.6, min=0.25, max=3.0, step=0.05, description="b:")
conv_button = widgets.Button(description="Convolve Gaussians", button_style="info", icon="link")
conv_output = widgets.Output()


def gaussian_convolution(_=None):
    """Compare FFT convolution, exact convolution, and spectral multiplication."""
    with conv_output:
        clear_output(wait=True)
        a, b = conv_a.value, conv_b.value
        t = time_grid(half_width=20.0, samples=2**14)
        dt = t[1] - t[0]
        f = np.exp(-a * t * t)
        g = np.exp(-b * t * t)
        omega, f_hat = continuous_fft(t, f)
        _, g_hat = continuous_fft(t, g)
        product = f_hat * g_hat
        numerical_conv = np.real(continuous_ifft(omega, product))
        exact_conv = np.sqrt(PI / (a + b)) * np.exp(-(a * b / (a + b)) * t * t)
        error = np.max(np.abs(numerical_conv - exact_conv))

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, f, label="f")
        axes[0].plot(t, g, label="g")
        axes[0].set_xlim(-6, 6)
        axes[0].set_title("Input Gaussians")
        axes[0].legend()
        axes[1].plot(t, numerical_conv, color="#2ca25f", label="FFT convolution")
        axes[1].plot(t, exact_conv, color="black", linestyle="--", label="exact")
        axes[1].set_xlim(-8, 8)
        axes[1].set_title("Convolution")
        axes[1].legend()
        plt.show()
        display(HTML(f"<b>Maximum convolution error:</b> {error:.3e}"))


conv_button.on_click(gaussian_convolution)
display(widgets.VBox([widgets.HBox([conv_a, conv_b]), conv_button, conv_output]))
gaussian_convolution()


## 8. Fourier inversion and frequency truncation

When inversion is justified,

$$
f(t)=\frac1{2\pi}\int_{-\infty}^{\infty}\widehat f(\omega)e^{i\omega t}\,d\omega.
$$

A numerical reconstruction uses only $|\omega|\le \Omega$:

$$
f_\Omega(t)=\frac1{2\pi}\int_{-\Omega}^{\Omega}\widehat f(\omega)e^{i\omega t}\,d\omega.
$$

For smooth functions, frequency truncation is efficient. For a rectangular pulse, the sharp jump requires arbitrarily high frequencies and produces oscillations analogous to the Gibbs phenomenon.


In [ ]:
# Reconstruct a function after discarding frequencies outside a chosen band.
inversion_function = widgets.Dropdown(options=["Rectangular pulse", "Gaussian"], description="Function:")
inversion_cutoff = widgets.FloatSlider(value=8.0, min=1.0, max=40.0, step=0.5, description="Omega:")
inversion_button = widgets.Button(description="Invert truncated spectrum", button_style="primary", icon="undo")
inversion_output = widgets.Output()


def truncated_inversion(_=None):
    """Apply an ideal low-pass cutoff and invert the sampled spectrum."""
    with inversion_output:
        clear_output(wait=True)
        t = time_grid(half_width=20.0, samples=2**14)
        if inversion_function.value == "Rectangular pulse":
            target = (np.abs(t) <= 1.0).astype(float)
        else:
            target = np.exp(-t * t)
        omega, transform = continuous_fft(t, target)
        retained = np.abs(omega) <= inversion_cutoff.value
        filtered = np.where(retained, transform, 0.0)
        reconstruction = np.real(continuous_ifft(omega, filtered))
        l2_error = l2_norm_grid(t, target - reconstruction)

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, target, color="black", linewidth=2, label="target")
        axes[0].plot(t, reconstruction, color="#1565c0", label="band-limited reconstruction")
        axes[0].set_xlim(-5, 5)
        axes[0].legend()
        axes[1].plot(omega, np.abs(transform), color="#d95f02")
        axes[1].axvspan(-inversion_cutoff.value, inversion_cutoff.value, color="#9ecae1", alpha=0.35)
        axes[1].set_xlim(-45, 45)
        axes[1].set_title("Retained frequency band")
        plt.show()
        display(HTML(f"<b>Finite-grid L2 reconstruction error:</b> {l2_error:.6f}"))


inversion_button.on_click(truncated_inversion)
display(widgets.VBox([widgets.HBox([inversion_function, inversion_cutoff]), inversion_button, inversion_output]))
truncated_inversion()


## 9. Plancherel and the uncertainty principle

With our normalization, Plancherel's identity is

$$
\int_{-\infty}^{\infty}|f(t)|^2\,dt
=\frac1{2\pi}\int_{-\infty}^{\infty}|\widehat f(\omega)|^2\,d\omega.
$$

If both first moments vanish and the variances are finite, the Heisenberg inequality gives

$$
\sigma_t\sigma_\omega\ge\frac12.
$$

Gaussian functions attain equality. Adding a quadratic phase (a *chirp*) leaves $|f(t)|$ unchanged but spreads the spectrum, making the inequality strict.


In [ ]:
# Check Plancherel and measure time-frequency uncertainty for a chirped Gaussian.
uncertainty_width = widgets.FloatSlider(value=1.0, min=0.25, max=3.0, step=0.05, description="a:")
uncertainty_chirp = widgets.FloatSlider(value=0.0, min=0.0, max=3.0, step=0.05, description="chirp b:")
uncertainty_button = widgets.Button(description="Measure energy and spread", button_style="success", icon="balance-scale")
uncertainty_output = widgets.Output()


def check_uncertainty(_=None):
    """Evaluate Plancherel energies and normalized variances."""
    with uncertainty_output:
        clear_output(wait=True)
        a, b = uncertainty_width.value, uncertainty_chirp.value
        t = time_grid(half_width=18.0, samples=2**15)
        f = np.exp(-(a - 1j * b) * t * t)
        omega, transform = continuous_fft(t, f)
        time_energy = np.real(integrate_grid(t, np.abs(f) ** 2))
        frequency_energy = np.real(integrate_grid(omega, np.abs(transform) ** 2)) / TWO_PI
        time_variance = np.real(integrate_grid(t, t * t * np.abs(f) ** 2)) / time_energy
        frequency_variance = (
            np.real(integrate_grid(omega, omega * omega * np.abs(transform) ** 2))
            / (TWO_PI * frequency_energy)
        )
        product = np.sqrt(time_variance * frequency_variance)

        central = np.abs(omega) <= 15
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, np.abs(f) ** 2, color="#1565c0")
        axes[0].set_xlim(-6, 6)
        axes[0].set_title("Time energy density")
        axes[1].plot(omega[central], np.abs(transform[central]) ** 2, color="#d95f02")
        axes[1].set_title("Frequency energy density")
        plt.show()

        display(HTML(
            f"<b>Time energy:</b> {time_energy:.9f}<br>"
            f"<b>Frequency energy / 2π:</b> {frequency_energy:.9f}<br>"
            f"<b>σ_t σ_ω:</b> {product:.9f} &nbsp; (lower bound = 0.5)"
        ))


uncertainty_button.on_click(check_uncertainty)
display(widgets.VBox([widgets.HBox([uncertainty_width, uncertainty_chirp]), uncertainty_button, uncertainty_output]))
check_uncertainty()


## 10. Gaussian approximate identities and heat flow

The heat kernel is

$$
G_s(t)=\frac1{\sqrt{4\pi s}}e^{-t^2/(4s)},
\qquad
\widehat{G_s}(\omega)=e^{-s\omega^2}.
$$

It has unit mass and $G_s*f\to f$ as $s\downarrow0$ in the appropriate sense. At the same time,

$$
u(t,s)=(G_s*f)(t)
$$

solves $u_s=u_{tt}$. Frequency $\omega$ is damped by $e^{-s\omega^2}$, so heat flow is a natural low-pass filter.


In [ ]:
# Apply the Gaussian heat multiplier to a nonsmooth initial profile.
heat_initial = widgets.Dropdown(options=["Box", "Two separated pulses", "Noisy Gaussian"], description="Initial data:")
heat_time = widgets.FloatSlider(value=0.15, min=0.01, max=1.0, step=0.01, description="s:")
heat_play = widgets.Play(value=15, min=1, max=100, step=1, interval=100, description="Play")
heat_output = widgets.Output()


def set_heat_time(change):
    """Convert the play position to heat time."""
    heat_time.value = change["new"] / 100.0


def draw_heat_flow(change=None):
    """Evolve initial data by multiplying its transform by exp(-s omega^2)."""
    with heat_output:
        clear_output(wait=True)
        t = time_grid(half_width=20.0, samples=2**14)
        if heat_initial.value == "Box":
            initial = (np.abs(t) <= 2.0).astype(float)
        elif heat_initial.value == "Two separated pulses":
            initial = np.exp(-3 * (t - 3) ** 2) + 0.8 * np.exp(-5 * (t + 3) ** 2)
        else:
            initial = np.exp(-0.4 * t * t) * (1 + 0.30 * np.cos(9 * t) + 0.20 * np.sin(14 * t))

        omega, initial_hat = continuous_fft(t, initial)
        multiplier = np.exp(-heat_time.value * omega * omega)
        evolved = np.real(continuous_ifft(omega, initial_hat * multiplier))

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, initial, color="black", linestyle="--", label="initial")
        axes[0].plot(t, evolved, color="#d7301f", linewidth=2, label="heat-evolved")
        axes[0].set_xlim(-8, 8)
        axes[0].legend()
        axes[1].plot(omega, np.abs(initial_hat), color="#636363", label="initial spectrum")
        axes[1].plot(omega, np.abs(initial_hat * multiplier), color="#2ca25f", label="damped spectrum")
        axes[1].set_xlim(-20, 20)
        axes[1].legend()
        plt.show()

        mass_initial = np.real(integrate_grid(t, initial))
        mass_evolved = np.real(integrate_grid(t, evolved))
        display(HTML(f"<b>Mass check:</b> initial = {mass_initial:.7f}, evolved = {mass_evolved:.7f}"))


heat_play.observe(set_heat_time, names="value")
heat_initial.observe(draw_heat_flow, names="value")
heat_time.observe(draw_heat_flow, names="value")
display(widgets.VBox([widgets.HBox([heat_initial, heat_play, heat_time]), heat_output]))
draw_heat_flow()


## 11. Characteristic functions: probability as Fourier analysis

For a random variable $X$, its characteristic function is

$$
\varphi_X(u)=\mathbb E[e^{iuX}].
$$

It always exists because $|e^{iuX}|=1$. If $X$ has density $p$, then

$$
\varphi_X(u)=\int_{-\infty}^{\infty}p(x)e^{iux}\,dx
=\widehat p(-u).
$$

In particular, $\varphi_X(0)=1$, $|\varphi_X(u)|\le1$, and $\varphi_X(-u)=\overline{\varphi_X(u)}$.


In [ ]:
# Explore exact characteristic functions of standard distributions.
cf_distribution = widgets.Dropdown(
    options=["Bernoulli", "Binomial", "Poisson", "Normal", "Exponential", "Uniform", "Cauchy"],
    value="Normal",
    description="Distribution:",
)
cf_parameter = widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1, description="Parameter:")
cf_button = widgets.Button(description="Plot characteristic function", button_style="primary", icon="chart-line")
cf_output = widgets.Output()


def characteristic_function(name, u, parameter):
    """Return a standard characteristic function with one adjustable parameter."""
    if name == "Bernoulli":
        p = min(parameter / 4.0, 0.95)
        return 1 - p + p * np.exp(1j * u), f"p={p:.2f}"
    if name == "Binomial":
        p, n = 0.35, max(1, int(round(parameter * 3)))
        return (1 - p + p * np.exp(1j * u)) ** n, f"n={n}, p={p}"
    if name == "Poisson":
        lam = parameter
        return np.exp(lam * (np.exp(1j * u) - 1)), f"lambda={lam:.2f}"
    if name == "Normal":
        sigma = parameter
        return np.exp(-0.5 * sigma * sigma * u * u), f"mean=0, sigma={sigma:.2f}"
    if name == "Exponential":
        rate = parameter
        return rate / (rate - 1j * u), f"rate={rate:.2f}"
    if name == "Uniform":
        width = parameter
        return np.sinc(width * u / PI), f"Uniform[-{width:.2f},{width:.2f}]"
    scale = parameter
    return np.exp(-scale * np.abs(u)), f"scale={scale:.2f}"


def show_characteristic_function(_=None):
    """Plot real, imaginary, and absolute parts and verify basic bounds."""
    with cf_output:
        clear_output(wait=True)
        u = np.linspace(-12, 12, 2401)
        phi, parameters = characteristic_function(cf_distribution.value, u, cf_parameter.value)

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(u, np.real(phi), label="Re phi")
        axes[0].plot(u, np.imag(phi), label="Im phi")
        axes[0].set_title("Complex components")
        axes[0].legend()
        axes[1].plot(u, np.abs(phi), color="#d95f02")
        axes[1].axhline(1, color="black", linestyle="--")
        axes[1].set_title("Magnitude")
        plt.show()
        phi_zero, _ = characteristic_function(cf_distribution.value, np.array([0.0]), cf_parameter.value)
        display(HTML(
            f"<b>{parameters}</b><br>phi(0) = {phi_zero[0]:.6g}; "
            f"maximum sampled |phi| = {np.max(np.abs(phi)):.9f}."
        ))


cf_button.on_click(show_characteristic_function)
display(widgets.VBox([widgets.HBox([cf_distribution, cf_parameter]), cf_button, cf_output]))
show_characteristic_function()


## 12. Independent sums and moments from derivatives

If $X$ and $Y$ are independent, then

$$
\varphi_{X+Y}(u)=\varphi_X(u)\varphi_Y(u).
$$

For iid variables $X_1,\ldots,X_n$,

$$
\varphi_{X_1+\cdots+X_n}(u)=\varphi_X(u)^n.
$$

When the corresponding moments exist,

$$
\varphi_X'(0)=i\mathbb E[X],
\qquad
\varphi_X''(0)=-\mathbb E[X^2].
$$

The simulation uses exponential variables. Their sum has a Gamma distribution and characteristic function $(\lambda/(\lambda-iu))^n$.


In [ ]:
# Verify independent-sum multiplication and recover moments numerically.
sum_count = widgets.IntSlider(value=4, min=1, max=20, step=1, description="n:")
sum_rate = widgets.FloatSlider(value=1.5, min=0.5, max=4.0, step=0.1, description="rate:")
sum_button = widgets.Button(description="Simulate independent sum", button_style="info", icon="plus")
sum_output = widgets.Output()


def independent_sum_experiment(_=None):
    """Compare empirical and exact characteristic functions for a Gamma sum."""
    with sum_output:
        clear_output(wait=True)
        n, rate = sum_count.value, sum_rate.value
        rng = np.random.default_rng(20260717 + n)
        samples = rng.exponential(1 / rate, size=(60000, n)).sum(axis=1)
        u = np.linspace(-5, 5, 501)

        # Compute the empirical characteristic function in moderate-size chunks.
        empirical = np.array([np.mean(np.exp(1j * value * samples)) for value in u])
        exact = (rate / (rate - 1j * u)) ** n
        max_error = np.max(np.abs(empirical - exact))

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].hist(samples, bins=80, density=True, color="#9ecae1", edgecolor="white")
        axes[0].set_title("Simulated sum")
        axes[1].plot(u, np.real(exact), color="black", label="exact Re phi")
        axes[1].plot(u, np.real(empirical), color="#d95f02", alpha=0.8, label="empirical Re phi")
        axes[1].legend()
        axes[1].set_title("Product rule for independent sums")
        plt.show()

        theoretical_mean = n / rate
        theoretical_variance = n / rate**2
        display(HTML(
            f"<b>Mean:</b> empirical {np.mean(samples):.5f}, exact {theoretical_mean:.5f}<br>"
            f"<b>Variance:</b> empirical {np.var(samples):.5f}, exact {theoretical_variance:.5f}<br>"
            f"<b>Maximum empirical CF error:</b> {max_error:.4f}"
        ))


sum_button.on_click(independent_sum_experiment)
display(widgets.VBox([widgets.HBox([sum_count, sum_rate]), sum_button, sum_output]))
independent_sum_experiment()


## 13. Central limit theorem through characteristic functions

Let $X_1,X_2,\ldots$ be iid with mean $\mu$ and variance $\sigma^2>0$. Then

$$
Z_n=\frac{X_1+\cdots+X_n-n\mu}{\sigma\sqrt n}
\xrightarrow{d}N(0,1).
$$

At the characteristic-function level,

$$
\varphi_{Z_n}(u)
=\left[
e^{-iu\mu/(\sigma\sqrt n)}
\varphi_X\left(\frac{u}{\sigma\sqrt n}\right)
\right]^n
\longrightarrow e^{-u^2/2}.
$$

Choose a non-Gaussian parent distribution and watch both its standardized sums and their characteristic functions approach the normal law.


In [ ]:
# Visualize the central limit theorem in density and frequency domains.
clt_parent = widgets.Dropdown(options=["Centered exponential", "Uniform", "Rademacher"], description="Parent:")
clt_count = widgets.IntSlider(value=8, min=1, max=80, step=1, description="n:")
clt_button = widgets.Button(description="Run CLT experiment", button_style="success", icon="chart-bar")
clt_output = widgets.Output()


def clt_experiment(_=None):
    """Simulate standardized sums and compare characteristic functions."""
    with clt_output:
        clear_output(wait=True)
        n = clt_count.value
        rng = np.random.default_rng(6100 + n)
        repetitions = 50000

        if clt_parent.value == "Centered exponential":
            raw = rng.exponential(1.0, size=(repetitions, n)) - 1.0
            standardized_sum = raw.sum(axis=1) / np.sqrt(n)
            base_cf = lambda v: np.exp(-1j * v) / (1 - 1j * v)
        elif clt_parent.value == "Uniform":
            raw = rng.uniform(-np.sqrt(3), np.sqrt(3), size=(repetitions, n))
            standardized_sum = raw.sum(axis=1) / np.sqrt(n)
            base_cf = lambda v: np.sinc(np.sqrt(3) * v / PI)
        else:
            raw = rng.choice([-1.0, 1.0], size=(repetitions, n))
            standardized_sum = raw.sum(axis=1) / np.sqrt(n)
            base_cf = lambda v: np.cos(v)

        u = np.linspace(-6, 6, 1001)
        exact_sum_cf = base_cf(u / np.sqrt(n)) ** n
        normal_cf = np.exp(-u * u / 2)
        x = np.linspace(-4, 4, 600)
        normal_density = np.exp(-x * x / 2) / np.sqrt(TWO_PI)

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].hist(standardized_sum, bins=80, density=True, color="#9ecae1", edgecolor="white")
        axes[0].plot(x, normal_density, color="#c43c39", linewidth=2, label="N(0,1)")
        axes[0].set_xlim(-4, 4)
        axes[0].legend()
        axes[0].set_title("Standardized sum")
        axes[1].plot(u, np.real(exact_sum_cf), color="#1565c0", label="sum CF")
        axes[1].plot(u, normal_cf, color="black", linestyle="--", label="exp(-u^2/2)")
        axes[1].legend()
        axes[1].set_title("Characteristic-function convergence")
        plt.show()

        cf_error = np.max(np.abs(exact_sum_cf - normal_cf))
        display(HTML(f"<b>Maximum displayed characteristic-function error:</b> {cf_error:.5f}"))


clt_button.on_click(clt_experiment)
display(widgets.VBox([widgets.HBox([clt_parent, clt_count]), clt_button, clt_output]))
clt_experiment()


## 14. Solving a constant-coefficient equation

Consider

$$
-u''(t)+m^2u(t)=f(t),
\qquad f(t)=e^{-t^2}.
$$

Formally transforming gives

$$
(\omega^2+m^2)\widehat u(\omega)=\widehat f(\omega),
\qquad
\widehat u(\omega)=\frac{\widehat f(\omega)}{\omega^2+m^2}.
$$

After inverse transformation we obtain a candidate $u$. The decisive final step is to verify the original equation. The code computes $u$, differentiates spectrally, and reports the residual

$$
r=-u''+m^2u-f.
$$


In [ ]:
# Solve -u'' + m^2 u = exp(-t^2) and verify the original equation.
ode_mass = widgets.FloatSlider(value=1.0, min=0.25, max=3.0, step=0.05, description="m:")
ode_button = widgets.Button(description="Solve and verify", button_style="primary", icon="check-double")
ode_output = widgets.Output()


def solve_transform_ode(_=None):
    """Construct the spectral candidate and directly verify its residual."""
    with ode_output:
        clear_output(wait=True)
        m = ode_mass.value
        t = time_grid(half_width=24.0, samples=2**14)
        forcing = np.exp(-t * t)
        omega, forcing_hat = continuous_fft(t, forcing)
        candidate_hat = forcing_hat / (omega * omega + m * m)
        candidate = np.real(continuous_ifft(omega, candidate_hat))

        # In frequency space, the transform of -u'' is omega^2 * U.
        minus_second_derivative = np.real(continuous_ifft(omega, omega * omega * candidate_hat))
        residual = minus_second_derivative + m * m * candidate - forcing
        residual_l2 = l2_norm_grid(t, residual)

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, forcing, color="black", linestyle="--", label="forcing f")
        axes[0].plot(t, candidate, color="#1565c0", label="candidate u")
        axes[0].set_xlim(-8, 8)
        axes[0].legend()
        axes[1].plot(t, residual, color="#c43c39")
        axes[1].set_xlim(-8, 8)
        axes[1].set_title("Direct equation residual")
        plt.show()
        display(HTML(f"<b>Residual L2 norm:</b> {residual_l2:.3e}"))


ode_button.on_click(solve_transform_ode)
display(widgets.VBox([ode_mass, ode_button, ode_output]))
solve_transform_ode()


## 15. Formal transform calculations beyond classical hypotheses

A transform calculation can be used as a **discovery device** even when the unknown function is not known to lie in $L^1$ or $L^2$. The logically safe workflow is:

$$
\boxed{
\text{formal transform}
\longrightarrow
\text{candidate formula}
\longrightarrow
\text{direct verification in the original problem}
}
$$

Three examples from the chapter are included:

$$
\begin{aligned}
-u''+a^2u&=\cos(bx),\\
u_t&=\kappa u_{xx},\qquad u(x,0)=\cos(bx),\\
u-k*u&=\cos(bx),\qquad k(x)=\tfrac12e^{-|x|}.
\end{aligned}
$$

Their candidates are bounded oscillatory functions outside both $L^1(\mathbb R)$ and $L^2(\mathbb R)$. The code checks the original equation directly, without claiming that the classical transform of $\cos(bx)$ exists.


In [ ]:
# Directly verify several valid solutions outside classical L1 and L2.
formal_problem = widgets.Dropdown(options=["Oscillatory ODE", "Heat equation", "Convolution equation"], description="Problem:")
formal_a = widgets.FloatSlider(value=1.2, min=0.2, max=3.0, step=0.1, description="a:")
formal_b = widgets.FloatSlider(value=1.5, min=0.2, max=4.0, step=0.1, description="b:")
formal_kappa = widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.1, description="kappa:")
formal_time = widgets.FloatSlider(value=0.6, min=0.0, max=3.0, step=0.05, description="time:")
formal_button = widgets.Button(description="Verify candidate directly", button_style="warning", icon="clipboard-check")
formal_output = widgets.Output()


def verify_formal_candidate(_=None):
    """Check the selected original equation analytically on a grid."""
    with formal_output:
        clear_output(wait=True)
        a, b = formal_a.value, formal_b.value
        kappa, time = formal_kappa.value, formal_time.value
        x = np.linspace(-4 * PI, 4 * PI, 4001)

        if formal_problem.value == "Oscillatory ODE":
            forcing = np.cos(b * x)
            candidate = forcing / (a * a + b * b)
            second_derivative = -b * b * candidate
            residual = -second_derivative + a * a * candidate - forcing
            auxiliary_residual = np.zeros_like(x)
            formula = "u(x)=cos(bx)/(a^2+b^2)"
        elif formal_problem.value == "Heat equation":
            forcing = np.cos(b * x)
            candidate = np.exp(-kappa * b * b * time) * forcing
            time_derivative = -kappa * b * b * candidate
            second_space_derivative = -b * b * candidate
            residual = time_derivative - kappa * second_space_derivative
            auxiliary_residual = np.cos(b * x) - forcing
            formula = "u(x,t)=exp(-kappa*b^2*t) cos(bx)"
        else:
            forcing = np.cos(b * x)
            candidate = (1 + b * b) * forcing / (b * b)
            # Direct integration gives k*cos(b·)=cos(bx)/(1+b^2).
            kernel_convolution = candidate / (1 + b * b)
            residual = candidate - kernel_convolution - forcing
            auxiliary_residual = np.zeros_like(x)
            formula = "u(x)=(1+b^2)cos(bx)/b^2"

        fig, ax = plt.subplots()
        ax.plot(x, forcing, color="black", linestyle="--", label="forcing or initial profile")
        ax.plot(x, candidate, color="#d95f02", label="candidate")
        ax.set_xlim(-2 * PI, 2 * PI)
        ax.legend()
        ax.set_title("A classical solution outside spatial L1 and L2")
        plt.show()

        display(HTML(
            f"<b>Candidate:</b> {formula}<br>"
            f"<b>Maximum equation residual:</b> {np.max(np.abs(residual)):.3e}<br>"
            f"<b>Maximum auxiliary-condition residual:</b> {np.max(np.abs(auxiliary_residual)):.3e}<br>"
            "<i>The candidate is accepted because it solves the original problem, not because cos(bx) satisfies a classical Fourier-transform hypothesis.</i>"
        ))


formal_button.on_click(verify_formal_candidate)
display(widgets.VBox([
    formal_problem,
    widgets.HBox([formal_a, formal_b]),
    widgets.HBox([formal_kappa, formal_time]),
    formal_button,
    formal_output,
]))
verify_formal_candidate()


## 16. A convolution integral equation

Consider

$$
u-\lambda(k*u)=f,
\qquad
k(t)=e^{-|t|},
\qquad
f(t)=e^{-t^2}.
$$

Since

$$
\widehat k(\omega)=\frac2{1+\omega^2},
$$

the transformed equation gives

$$
\widehat u(\omega)
=\frac{\widehat f(\omega)}{1-\lambda\widehat k(\omega)}.
$$

For $0\le\lambda<1/2$, the denominator never vanishes. The code constructs $u$ and independently evaluates $u-\lambda k*u-f$.


In [ ]:
# Solve a convolution equation and directly check its residual.
integral_lambda = widgets.FloatSlider(value=0.25, min=0.0, max=0.48, step=0.01, description="lambda:")
integral_button = widgets.Button(description="Solve integral equation", button_style="info", icon="project-diagram")
integral_output = widgets.Output()


def solve_integral_equation(_=None):
    """Solve u-lambda*k*u=f spectrally and verify the equation."""
    with integral_output:
        clear_output(wait=True)
        lam = integral_lambda.value
        t = time_grid(half_width=30.0, samples=2**15)
        forcing = np.exp(-t * t)
        omega, forcing_hat = continuous_fft(t, forcing)
        kernel_hat = 2.0 / (1.0 + omega * omega)
        denominator = 1.0 - lam * kernel_hat
        candidate_hat = forcing_hat / denominator
        candidate = np.real(continuous_ifft(omega, candidate_hat))
        convolution = np.real(continuous_ifft(omega, kernel_hat * candidate_hat))
        residual = candidate - lam * convolution - forcing
        residual_norm = l2_norm_grid(t, residual)

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, forcing, color="black", linestyle="--", label="f")
        axes[0].plot(t, candidate, color="#1565c0", label="u")
        axes[0].set_xlim(-8, 8)
        axes[0].legend()
        axes[1].plot(t, residual, color="#c43c39")
        axes[1].set_xlim(-8, 8)
        axes[1].set_title("Direct integral-equation residual")
        plt.show()

        display(HTML(
            f"<b>Minimum spectral denominator:</b> {np.min(denominator):.5f}<br>"
            f"<b>Residual L2 norm:</b> {residual_norm:.3e}"
        ))


integral_button.on_click(solve_integral_equation)
display(widgets.VBox([integral_lambda, integral_button, integral_output]))
solve_integral_equation()


## 17. Theorem and method selector

Different questions require different hypotheses. Before using a transform theorem, identify the intended conclusion:

$$
\text{existence and continuity},\quad
\text{decay},\quad
\text{inversion},\quad
\text{energy preservation},\quad
\text{probability convergence},\quad
\text{candidate discovery}.
$$

The last category is deliberately different: a formal calculation may suggest a candidate, but the final justification comes from direct substitution into the original problem.


In [ ]:
# Recommend the relevant theorem or validation method for a selected goal.
method_data = {
    "Show the transform is bounded and continuous": (
        "L1 boundedness and uniform continuity",
        "Assume f belongs to L1(R). Then |f-hat(omega)| <= ||f||_1 and f-hat is uniformly continuous."
    ),
    "Show the spectrum vanishes at infinity": (
        "Riemann--Lebesgue lemma",
        "Assume f belongs to L1(R). Then f-hat(omega) tends to zero as |omega| tends to infinity."
    ),
    "Recover a function pointwise": (
        "Pointwise Fourier inversion",
        "Use the chapter's local regularity and integrability hypotheses; at a jump expect the midpoint of the one-sided limits."
    ),
    "Preserve quadratic energy": (
        "Parseval--Plancherel theorem",
        "Extend the transform to L2 and use ||f||_2^2=(1/2pi)||f-hat||_2^2."
    ),
    "Analyze an independent sum": (
        "Characteristic-function product rule",
        "Independence gives phi_(X+Y)=phi_X phi_Y."
    ),
    "Prove convergence in distribution": (
        "Levy continuity theorem",
        "Show pointwise convergence of characteristic functions to a function continuous at zero."
    ),
    "Use a transform outside classical hypotheses": (
        "Formal discovery plus direct verification",
        "Derive a candidate formally, then check the original equation, boundary/initial conditions, and desired regularity directly."
    ),
}

method_goal = widgets.Dropdown(options=list(method_data), description="Goal:", layout=widgets.Layout(width="85%"))
method_button = widgets.Button(description="Select method", button_style="primary", icon="book-open")
method_output = widgets.Output()


def select_method(_=None):
    """Display the recommended theorem and its logical role."""
    with method_output:
        clear_output(wait=True)
        title, explanation = method_data[method_goal.value]
        display(HTML(
            f"<div style='padding:12px;border:1px solid #aaccee;border-radius:6px'>"
            f"<h4 style='margin-top:0'>{title}</h4>{explanation}</div>"
        ))


method_button.on_click(select_method)
display(widgets.VBox([method_goal, method_button, method_output]))
select_method()


## 18. Concept quiz

The quiz emphasizes normalization and logical validity:

- the transform of a derivative carries the factor $i\omega$;
- Plancherel contains $1/(2\pi)$ under this convention;
- convolution becomes multiplication;
- characteristic functions always exist;
- a formal transform step is not, by itself, a proof that the candidate solves the original problem.

Choose one response for each question and press **Check answers**.


In [ ]:
# Self-checking quiz with explanations.
quiz_questions = [
    ("Under this convention, F{f'} equals:", ["omega F", "i omega F", "-i omega F"], 1,
     "Integration by parts gives the factor i omega."),
    ("The transform of a time translation f(t-t0) is:", ["F(omega-t0)", "exp(-i omega t0)F(omega)", "exp(i omega t0)F(omega)"], 1,
     "Time translation produces the phase exp(-i omega t0)."),
    ("Convolution in time becomes:", ["multiplication in frequency", "differentiation in frequency", "translation in frequency"], 0,
     "The convolution theorem gives F{f*g}=F{f}F{g}."),
    ("Plancherel's identity here contains:", ["no constant", "a factor 1/(2pi) on spectral energy", "a factor 2pi on spectral energy"], 1,
     "Our forward transform has no prefactor, so spectral energy is divided by 2pi."),
    ("A characteristic function:", ["exists only when a density exists", "exists only when moments exist", "always exists"], 2,
     "The integrand exp(iuX) is bounded by one."),
    ("Pointwise convergence of characteristic functions is useful for:", ["Levy's continuity theorem", "only computing moments", "only discrete random variables"], 0,
     "Levy's theorem turns suitable CF convergence into convergence in distribution."),
    ("A formally derived candidate outside L1 and L2 is valid when:", ["the algebra looks plausible", "it directly satisfies the original problem and required conditions", "it has a graph"], 1,
     "Direct verification of the equation and auxiliary conditions supplies the justification."),
]

quiz_boxes = []
for number, (question, options, _, _) in enumerate(quiz_questions, start=1):
    radio = widgets.RadioButtons(options=options, description=f"{number}.", layout=widgets.Layout(width="95%"))
    quiz_boxes.append(widgets.VBox([widgets.HTML(f"<b>{question}</b>"), radio]))

quiz_button = widgets.Button(description="Check answers", button_style="success", icon="check-circle")
quiz_output = widgets.Output()


def grade_quiz(_=None):
    """Grade the conceptual quiz and explain incorrect answers."""
    with quiz_output:
        clear_output(wait=True)
        score = 0
        feedback = []
        for number, ((question, options, correct, explanation), box) in enumerate(zip(quiz_questions, quiz_boxes), start=1):
            selected = box.children[1].value
            if selected == options[correct]:
                score += 1
                feedback.append(f"<span style='color:#198754'>✓ {number}. Correct.</span>")
            else:
                feedback.append(f"<span style='color:#b02a37'>✗ {number}. {explanation}</span>")
        display(HTML(f"<h4>Score: {score}/{len(quiz_questions)}</h4>" + "<br>".join(feedback)))


quiz_button.on_click(grade_quiz)
display(widgets.VBox(quiz_boxes + [quiz_button, quiz_output]))


## 19. Practice generator

Each problem should be solved in three layers whenever appropriate:

1. state the transform identity and its hypotheses;
2. carry out the frequency-domain calculation;
3. invert and verify the result in the original variables.

For probability problems, also distinguish existence of the characteristic function from existence of its derivatives and moments.


In [ ]:
# Generate a practice problem and reveal a compact solution guide.
practice_bank = {
    "Transform pairs": [
        ("Use scaling to transform the indicator of [-a,a].",
         "Starting from the unit pulse, scaling gives 2 sin(a omega)/omega, with value 2a at omega=0 by continuity."),
        ("Transform exp(-a|t|), a>0.",
         "Split at zero or use evenness: 2 integral_0^infinity exp(-at)cos(omega t)dt = 2a/(a^2+omega^2)."),
    ],
    "Rules": [
        ("Find the transform of exp(i omega0 t)f(t-t0).",
         "Translate first and modulate: exp[-i(omega-omega0)t0] F(omega-omega0)."),
        ("Express the transform of t f'(t) using F and its derivative.",
         "First F{f'}=i omega F. Then F{t f'}=i d/domega[i omega F]=-(F+omega F')."),
    ],
    "Probability": [
        ("Find the characteristic function of a sum of independent Poisson variables.",
         "Multiply exp(lambda_j(e^{iu}-1)); the result is Poisson with parameter sum lambda_j."),
        ("Recover mean and variance from phi'(0) and phi''(0).",
         "E[X]=phi'(0)/i and E[X^2]=-phi''(0); then Var(X)=E[X^2]-E[X]^2."),
    ],
    "Applications": [
        ("Solve formally -u''+m^2u=f in transform space.",
         "Set U=F/(omega^2+m^2), invert, then verify -u''+m^2u=f and the required decay or boundary conditions."),
        ("Explain how to use a formal transform when the candidate is not in L1 or L2.",
         "Treat the transform algebra as a discovery step only. Prove validity by direct differentiation/integration and by checking all original conditions."),
    ],
}

practice_level = widgets.Dropdown(options=list(practice_bank), description="Topic:")
practice_new = widgets.Button(description="New problem", button_style="primary", icon="random")
practice_reveal = widgets.Button(description="Reveal guide", button_style="warning", icon="lightbulb")
practice_problem = widgets.HTMLMath()
practice_solution = widgets.HTMLMath()
practice_state = {"solution": ""}
practice_rng = np.random.default_rng(20260717)


def new_practice_problem(_=None):
    """Choose a problem from the selected topic."""
    choices = practice_bank[practice_level.value]
    index = int(practice_rng.integers(0, len(choices)))
    problem, solution = choices[index]
    practice_state["solution"] = solution
    practice_problem.value = f"<div style='padding:12px;background:#eef6ff'><b>Problem.</b> {problem}</div>"
    practice_solution.value = ""


def reveal_practice_solution(_=None):
    """Reveal the current solution guide."""
    practice_solution.value = (
        "<div style='padding:12px;background:#fff4d6;border-left:4px solid #e0a800'>"
        f"<b>Solution guide.</b> {practice_state['solution']}</div>"
    )


practice_new.on_click(new_practice_problem)
practice_reveal.on_click(reveal_practice_solution)
practice_level.observe(new_practice_problem, names="value")
display(widgets.VBox([
    widgets.HBox([practice_level, practice_new, practice_reveal]),
    practice_problem,
    practice_solution,
]))
new_practice_problem()


## Suggested learning path and final checkpoint

1. Verify the five basic transform pairs before using any operational rule.
2. Use translation, modulation, and scaling to predict spectra without recomputing integrals.
3. Connect convolution, heat flow, and frequency damping.
4. Check Plancherel numerically and identify the Gaussian equality case in the uncertainty principle.
5. Interpret characteristic functions as Fourier transforms of probability laws.
6. Compare the CLT in both physical and frequency domains.
7. For every differential or integral equation, finish with a direct residual check.

The essential logical distinction is

$$
\text{a transform theorem justifies an operation under stated hypotheses},
$$

whereas

$$
\text{a formal calculation proposes a candidate whose validity must be checked directly}.
$$
